# Obesity Level Prediction

This notebook predicts obesity level using multiclass machine learning.

### Main idea
The notebook is intentionally written **without custom `def` functions** so the workflow is easy
to follow from top to bottom:

1. Load the data
2. Inspect and clean it
3. Explore the data
4. Create BMI
5. Split train/test data
6. Preprocess the features
7. Compare models with cross-validation
8. Tune the best models
9. Evaluate the final model
10. Save the model

Two feature tracks are compared:

- **Track A:** includes Height, Weight and BMI.
- **Track B:** removes Height, Weight and BMI and uses lifestyle features only.

> **Fixes applied to this version** (each one is marked `# FIX` in the code):
> 1. The data loader only looked in `/content/sample_data/`, so the notebook crashed with
>    `FileNotFoundError` anywhere else. It now searches several locations and, in Colab, offers
>    an upload prompt.
> 2. The BMI-reconstruction result was reported but never explained; the notebook now measures
>    *how far off* the disagreements are, which is what makes the leakage argument work.
> 3. Section 14 compared a **tuned** Track B model against an **untuned** Track A model. The
>    like-for-like number is now printed next to it.
> 4. The overfitting check called a purity-grown forest "substantial overfitting". It now reads
>    train / CV / test correctly for tree ensembles.
> 5. The saved `.pkl` was uncompressed (tens of MB). It is now compressed, and the feature list
>    is saved beside it so the model can actually be reused.
> **Final correction:** the generic 1-SE selector was removed because its tree-specific simplicity
> rules were not valid for SVM, Logistic Regression, or KNN. Final selection now uses the best
> cross-validated tuned model directly.


## 0. Setup

If needed, install the packages first:

```python
# %pip install pandas numpy matplotlib seaborn scikit-learn joblib
```


In [ ]:
import os
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV,
    RandomizedSearchCV,
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

import sklearn
print("Imports loaded successfully.")
print("pandas", pd.__version__, "| numpy", np.__version__, "| scikit-learn", sklearn.__version__)

### Visual Analysis

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.20
N_SPLITS = 5
TARGET = "NObeyesdad"

np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

print("Random state:", RANDOM_STATE)
print("Test size:", TEST_SIZE)
print("CV folds:", N_SPLITS)

## 1. Load the Dataset

Put `ObesityDataSet_raw_and_data_sinthetic.csv` next to the notebook, in a `data/` folder, or
upload it in Colab when prompted.


In [ ]:
                                                                                              
                                                                                   
                                                                                        
FILENAME = "ObesityDataSet_raw_and_data_sinthetic.csv"

CANDIDATE_PATHS = [
    FILENAME,                                                               
    f"data/{FILENAME}",                                            
    f"/content/{FILENAME}",                                                  
    f"/content/sample_data/{FILENAME}",                                  
    f"/content/drive/MyDrive/{FILENAME}",                                   
]

DATA_PATH = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)

                                                                     
if DATA_PATH is None:
    try:
        from google.colab import files
        print("Dataset not found. Choose the CSV file to upload:")
        uploaded = files.upload()
        DATA_PATH = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(
            "Dataset not found. Put the CSV in one of these locations:\n  "
            + "\n  ".join(CANDIDATE_PATHS)
        )

df = pd.read_csv(DATA_PATH)

print("Loaded from:", DATA_PATH)
print("Shape:", df.shape)

df.head()

## 2. Basic Inspection


In [ ]:
df.info()

### Distribution Analysis

In [ ]:
print("Missing values:")
display(df.isnull().sum().to_frame("Missing"))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTarget distribution:")
display(df[TARGET].value_counts().to_frame("Count"))

## 3. Correct Data Types

The yes/no columns are converted to boolean values.

Ordered categories such as `CAEC` and `CALC` will be handled later with `OrdinalEncoder`.


In [ ]:
BOOLEAN_COLS = [
    "family_history_with_overweight",
    "FAVC",
    "SMOKE",
    "SCC",
]

NOMINAL_COLS = [
    "Gender",
    "MTRANS",
]

ORDINAL_COLS = [
    "CAEC",
    "CALC",
]

for col in BOOLEAN_COLS:
    df[col] = df[col].map({"yes": True, "no": False}).astype("bool")

for col in NOMINAL_COLS + ORDINAL_COLS + [TARGET]:
    df[col] = df[col].astype("category")

                                                                                          
                                                                                         
                             
assert df[BOOLEAN_COLS].isnull().sum().sum() == 0, "Unexpected values in the yes/no columns"
print("Boolean mapping verified.")

df.dtypes.to_frame("Data Type")

## 4. Remove Duplicate Rows

Duplicates are removed **before** train/test splitting so the same row cannot appear in both
sets.


In [ ]:
before = len(df)
duplicates = df.duplicated().sum()

df = df.drop_duplicates().reset_index(drop=True)

print("Rows before:", before)
print("Duplicates removed:", duplicates)
print("Rows after:", len(df))

## 5. Exploratory Data Analysis


In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include=["category", "object"]).columns.tolist()
boolean_cols = df.select_dtypes(include="bool").columns.tolist()

print("Numeric columns:", numeric_cols)
print("\nCategorical columns:", categorical_cols)
print("\nBoolean columns:", boolean_cols)

### Summary Statistics

In [ ]:
df[numeric_cols].describe().T

### 5.1 Numeric Distributions


In [ ]:
n = len(numeric_cols)
rows = (n + 2) // 3

fig, axes = plt.subplots(rows, 3, figsize=(15, 3.5 * rows))
axes = np.array(axes).reshape(-1)

for ax, col in zip(axes, numeric_cols):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(col)

for ax in axes[n:]:
    ax.axis("off")

plt.tight_layout()
plt.show()

### 5.2 Categorical Distributions


In [ ]:
plot_cols = [c for c in categorical_cols if c != TARGET] + boolean_cols

rows = (len(plot_cols) + 2) // 3
fig, axes = plt.subplots(rows, 3, figsize=(15, 3.5 * rows))
axes = np.array(axes).reshape(-1)

for ax, col in zip(axes, plot_cols):
    sns.countplot(y=df[col].astype(str), ax=ax)
    ax.set_title(col)
    ax.set_ylabel("")

for ax in axes[len(plot_cols):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

### 5.3 Target Distribution


In [ ]:
CLASS_ORDER = [
    "Insufficient_Weight",
    "Normal_Weight",
    "Overweight_Level_I",
    "Overweight_Level_II",
    "Obesity_Type_I",
    "Obesity_Type_II",
    "Obesity_Type_III",
]

plt.figure(figsize=(10, 5))
sns.countplot(
    x=df[TARGET].astype(str),
    order=CLASS_ORDER
)
plt.xticks(rotation=45)
plt.title("Obesity Level Distribution")
plt.tight_layout()
plt.show()

### 5.4 Height and Weight vs Target

This plot matters because obesity level is strongly related to Height and Weight through BMI.


In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=df,
    x="Height",
    y="Weight",
    hue=df[TARGET].astype(str),
    hue_order=CLASS_ORDER,
    alpha=0.7
)

plt.title("Height vs Weight by Obesity Level")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 6. Feature Engineering — BMI

BMI is calculated as:

$$BMI = \frac{Weight}{Height^{2}}$$

*(Note: the original used `\[ ... \]`, which Jupyter and Colab do not render as math — it showed
up as raw text. `$$...$$` is the syntax that works.)*

The target classes are closely related to BMI ranges, so we also check how well BMI thresholds
alone can reconstruct the labels. `pd.cut()` does this without a function.


In [ ]:
df["BMI"] = df["Weight"] / (df["Height"] ** 2)

BMI_BINS = [
    -np.inf,
    18.5,
    25.0,
    27.5,
    30.0,
    35.0,
    40.0,
    np.inf,
]

reconstructed = pd.cut(
    df["BMI"],
    bins=BMI_BINS,
    labels=CLASS_ORDER,
    right=False,
)

agreement = (
    reconstructed.astype(str).values
    == df[TARGET].astype(str).values
).mean()

print(f"BMI threshold agreement with target: {agreement:.2%}")

df[["Height", "Weight", "BMI", TARGET]].head()

### 6.1 How wrong are the disagreements?

The agreement number on its own is not the argument. If the misses were scattered randomly
across the seven classes, the label would clearly depend on something other than BMI. If every
miss lands one band away, the label *is* a BMI rule and the residual is just boundary blur from
the synthetic rows.


In [ ]:
                                                                                 
class_position = {name: i for i, name in enumerate(CLASS_ORDER)}

actual_pos = df[TARGET].astype(str).map(class_position)
recon_pos = reconstructed.astype(str).map(class_position)
bands_apart = (actual_pos - recon_pos).abs()

wrong = bands_apart[bands_apart > 0]

print(f"Disagreements: {len(wrong)} of {len(df)}  ({len(wrong) / len(df):.2%})")
print(f"Exactly one band away: {(wrong == 1).mean():.1%} of them")
print()
print("Bands apart (0 = the BMI rule got it right):")
print(bands_apart.value_counts().sort_index().to_string())
print()
print("BMI range per class - notice each class occupies one contiguous interval:")
display(
    df.groupby(TARGET, observed=True)["BMI"]
      .agg(["min", "max", "count"])
      .round(2)
      .loc[CLASS_ORDER]
)

### Dataset Limitation

The dataset contains synthetic records. Some lifestyle features contain interpolated decimal
values, which is consistent with SMOTE-generated rows.

This means very high scores should be interpreted carefully.


In [ ]:
LIKERT_COLS = ["FCVC", "NCP", "CH2O", "FAF", "TUE"]

is_integral = np.isclose(
    df[LIKERT_COLS].to_numpy(),
    df[LIKERT_COLS].round().to_numpy()
).all(axis=1)

print("Likely original rows:", is_integral.sum())
print("Likely synthetic rows:", (~is_integral).sum())
print(f"Likely original percentage: {is_integral.mean():.1%}")

## 7. Create Two Feature Sets

**Track A** uses all features, including Height, Weight and BMI.

**Track B** removes Height, Weight and BMI to see how much information exists in the lifestyle
variables alone.


In [ ]:
LEAKY_FEATURES = ["Height", "Weight", "BMI"]

FEATURES_A = [col for col in df.columns if col != TARGET]
FEATURES_B = [col for col in FEATURES_A if col not in LEAKY_FEATURES]

print("Track A features:", len(FEATURES_A))
print(FEATURES_A)

print("\nTrack B features:", len(FEATURES_B))
print(FEATURES_B)

## 8. Train/Test Split

The split is done **once** and both tracks use the same rows.


In [ ]:
X = df[FEATURES_A].copy()
y = df[TARGET].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train_A = X_train[FEATURES_A]
X_test_A = X_test[FEATURES_A]

X_train_B = X_train[FEATURES_B]
X_test_B = X_test[FEATURES_B]

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Track A:", X_train_A.shape, X_test_A.shape)
print("Track B:", X_train_B.shape, X_test_B.shape)

## 9. Preprocessing

Instead of hiding preprocessing inside a function, the two preprocessors are written directly.

- Numeric → median imputation + standardization
- Nominal → most-frequent imputation + one-hot encoding
- Ordinal → most-frequent imputation + ordinal encoding
- Boolean → passed through directly


In [ ]:
CAEC_ORDER = ["no", "Sometimes", "Frequently", "Always"]
CALC_ORDER = ["no", "Sometimes", "Frequently", "Always"]

numeric_all = df.select_dtypes(include=np.number).columns.tolist()

num_A = [c for c in FEATURES_A if c in numeric_all]
nom_A = [c for c in FEATURES_A if c in NOMINAL_COLS]
ord_A = [c for c in FEATURES_A if c in ORDINAL_COLS]
bool_A = [c for c in FEATURES_A if c in BOOLEAN_COLS]

num_B = [c for c in FEATURES_B if c in numeric_all]
nom_B = [c for c in FEATURES_B if c in NOMINAL_COLS]
ord_B = [c for c in FEATURES_B if c in ORDINAL_COLS]
bool_B = [c for c in FEATURES_B if c in BOOLEAN_COLS]

                                                                                             
                                                                    
assert ord_A == ["CAEC", "CALC"] and ord_B == ["CAEC", "CALC"], f"Ordinal order changed: {ord_A}"

                                                                                           
                                                           
for label, feats, groups in [("A", FEATURES_A, [num_A, nom_A, ord_A, bool_A]),
                             ("B", FEATURES_B, [num_B, nom_B, ord_B, bool_B])]:
    routed = sum(len(g) for g in groups)
    assert routed == len(feats), f"Track {label}: {len(feats) - routed} feature(s) not routed"
    print(f"Track {label}: {routed}/{len(feats)} features routed")

print()
print("Track A -> numeric:", num_A)
print("            nominal:", nom_A, "| ordinal:", ord_A, "| boolean:", bool_A)
print("Track B -> numeric:", num_B)
print("            nominal:", nom_B, "| ordinal:", ord_B, "| boolean:", bool_B)

### Preprocessing Pipeline

In [ ]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

nominal_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

ordinal_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(
        categories=[CAEC_ORDER, CALC_ORDER],
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    )),
])

preprocess_A = ColumnTransformer([
    ("numeric", clone(numeric_pipe), num_A),
    ("nominal", clone(nominal_pipe), nom_A),
    ("ordinal", clone(ordinal_pipe), ord_A),
    ("boolean", "passthrough", bool_A),
], remainder="drop")

preprocess_B = ColumnTransformer([
    ("numeric", clone(numeric_pipe), num_B),
    ("nominal", clone(nominal_pipe), nom_B),
    ("ordinal", clone(ordinal_pipe), ord_B),
    ("boolean", "passthrough", bool_B),
], remainder="drop")

print("Preprocessors created.")

## 10. Define the Models

A dictionary is enough here; we do not need a `get_models()` function.


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    ),

    "K-Nearest Neighbors": KNeighborsClassifier(),

    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=RANDOM_STATE
    ),

    "Support Vector Machine": SVC(
        probability=True,
        random_state=RANDOM_STATE
    ),
}

print("Models:")
for name in models:
    print("-", name)

## 11. Cross-Validation

`F1 Macro` is the main metric because this is a multiclass problem and we want each class to
contribute equally.


In [ ]:
skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

SCORING = {
    "accuracy": "accuracy",
    "f1_macro": "f1_macro",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
}

print("Cross-validation ready.")

### 11.1 Track A — With Height, Weight and BMI


In [ ]:
results_A = []

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocess", clone(preprocess_A)),
        ("classifier", clone(model)),
    ])

    scores = cross_validate(
        pipeline,
        X_train_A,
        y_train,
        cv=skf,
        scoring=SCORING,
        n_jobs=-1,
        error_score="raise",                                                                 
    )

    results_A.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision Macro": scores["test_precision_macro"].mean(),
        "Recall Macro": scores["test_recall_macro"].mean(),
        "F1 Macro": scores["test_f1_macro"].mean(),
        "F1 Std": scores["test_f1_macro"].std(),
    })

    print(name, "F1 Macro =", round(scores["test_f1_macro"].mean(), 4))

cv_A = (
    pd.DataFrame(results_A)
    .sort_values("F1 Macro", ascending=False)
    .reset_index(drop=True)
)

cv_A.round(4)

### 11.2 Track B — Lifestyle Features Only


In [ ]:
results_B = []

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocess", clone(preprocess_B)),
        ("classifier", clone(model)),
    ])

    scores = cross_validate(
        pipeline,
        X_train_B,
        y_train,
        cv=skf,
        scoring=SCORING,
        n_jobs=-1,
        error_score="raise",
    )

    results_B.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision Macro": scores["test_precision_macro"].mean(),
        "Recall Macro": scores["test_recall_macro"].mean(),
        "F1 Macro": scores["test_f1_macro"].mean(),
        "F1 Std": scores["test_f1_macro"].std(),
    })

    print(name, "F1 Macro =", round(scores["test_f1_macro"].mean(), 4))

cv_B = (
    pd.DataFrame(results_B)
    .sort_values("F1 Macro", ascending=False)
    .reset_index(drop=True)
)

cv_B.round(4)

### 11.3 Compare Track A and Track B


In [ ]:
comparison = (
    cv_A[["Model", "F1 Macro"]]
    .rename(columns={"F1 Macro": "Track A F1"})
    .merge(
        cv_B[["Model", "F1 Macro"]]
        .rename(columns={"F1 Macro": "Track B F1"}),
        on="Model"
    )
)

comparison["Difference"] = (
    comparison["Track A F1"] - comparison["Track B F1"]
)

comparison = comparison.sort_values(
    "Track B F1",
    ascending=False
).reset_index(drop=True)

comparison.round(4)

### Visual Analysis

In [ ]:
plot_data = comparison.melt(
    id_vars="Model",
    value_vars=["Track A F1", "Track B F1"],
    var_name="Track",
    value_name="F1 Macro",
)

plt.figure(figsize=(11, 6))
sns.barplot(
    data=plot_data,
    x="F1 Macro",
    y="Model",
    hue="Track",
)

plt.title("Cross-Validation F1 Macro: Track A vs Track B")
plt.tight_layout()
plt.show()

## 12. Hyperparameter Tuning — Track B

To keep the notebook simple, only the **top 3 Track B models** are tuned.


In [ ]:
PARAM_GRIDS = {
    "Random Forest": {
        "classifier__n_estimators": [200, 400],
        "classifier__max_depth": [8, 12, 16, 20],
        "classifier__min_samples_split": [2, 5, 10],
        "classifier__min_samples_leaf": [1, 2, 4, 8],
        "classifier__max_features": ["sqrt", "log2"],
    },

    "Gradient Boosting": {
        "classifier__n_estimators": [150, 300],
        "classifier__learning_rate": [0.05, 0.1],
        "classifier__max_depth": [3, 5],
        "classifier__subsample": [0.8, 1.0],
    },

    "Support Vector Machine": {
        "classifier__C": [0.5, 1, 5, 10],
        "classifier__gamma": ["scale", 0.05, 0.1],
        "classifier__kernel": ["rbf"],
    },

    "Logistic Regression": {
        "classifier__C": [0.05, 0.1, 1, 5, 10],
        "classifier__penalty": ["l2"],
        "classifier__solver": ["lbfgs"],
    },

    "K-Nearest Neighbors": {
        "classifier__n_neighbors": [3, 5, 9, 15, 21],
        "classifier__weights": ["uniform", "distance"],
        "classifier__p": [1, 2],
    },

    "Decision Tree": {
        "classifier__max_depth": [5, 8, 12, 20],
        "classifier__min_samples_leaf": [2, 5, 10],
        "classifier__criterion": ["gini", "entropy"],
    },
}

TOP_K = 3
MAX_EXHAUSTIVE = 60
N_ITER = 40

top_models = cv_B["Model"].head(TOP_K).tolist()

print("Models selected for tuning:")
for name in top_models:
    grid_size = int(np.prod([len(v) for v in PARAM_GRIDS[name].values()]))
    method = "GridSearchCV" if grid_size <= MAX_EXHAUSTIVE else f"RandomizedSearchCV({N_ITER})"
    print(f"- {name:<24} {grid_size:>4} combinations -> {method}")

### Hyperparameter Tuning

In [ ]:
tuning_results = []
tuned_models = {}
searches = {}

for name in top_models:

    pipeline = Pipeline([
        ("preprocess", clone(preprocess_B)),
        ("classifier", clone(models[name])),
    ])

    grid = PARAM_GRIDS[name]
    grid_size = int(np.prod([len(values) for values in grid.values()]))

    if grid_size <= MAX_EXHAUSTIVE:
        search = GridSearchCV(
            pipeline,
            grid,
            cv=skf,
            scoring="f1_macro",
            n_jobs=-1,
            refit=True,
        )
        method = "GridSearchCV"

    else:
        search = RandomizedSearchCV(
            pipeline,
            grid,
            n_iter=min(N_ITER, grid_size),
            cv=skf,
            scoring="f1_macro",
            n_jobs=-1,
            random_state=RANDOM_STATE,
            refit=True,
        )
        method = "RandomizedSearchCV"

    print("Tuning:", name, "-", method)

    search.fit(X_train_B, y_train)

    baseline_f1 = cv_B.loc[
        cv_B["Model"] == name,
        "F1 Macro"
    ].iloc[0]

    tuned_models[name] = search.best_estimator_
    searches[name] = search

    tuning_results.append({
        "Model": name,
        "Baseline F1": baseline_f1,
        "Tuned F1": search.best_score_,
        "Improvement": search.best_score_ - baseline_f1,
        "Best Params": search.best_params_,
    })

    print(f"   tuned F1 = {search.best_score_:.4f}"
          f"   (baseline {baseline_f1:.4f}, change {search.best_score_ - baseline_f1:+.4f})")

tuning_df = (
    pd.DataFrame(tuning_results)
    .sort_values("Tuned F1", ascending=False)
    .reset_index(drop=True)
)

tuning_df[["Model", "Baseline F1", "Tuned F1", "Improvement"]].round(4)

### Results

In [ ]:
for i in range(len(tuning_df)):
    print("\n", tuning_df.loc[i, "Model"])
    print(tuning_df.loc[i, "Best Params"])

## 13. Select the Final Model

The final Track B model is selected using the **highest cross-validated F1 Macro score** from the
hyperparameter search.

This is intentionally simple and reliable: `GridSearchCV` / `RandomizedSearchCV` already refit
the best setting on the full training set when `refit=True`.

We do **not** apply a generic 1-SE simplicity rule here, because different model families define
"simplicity" differently. For example, tree depth is meaningful for a Random Forest but not for
SVM or Logistic Regression.


In [ ]:
best_model_name = tuning_df.loc[0, "Model"]
search = searches[best_model_name]

                                                                          
                                                           
final_model = search.best_estimator_
final_cv_f1 = float(search.best_score_)
final_params = dict(search.best_params_)

print("Selected model:", best_model_name)
print(f"Best tuned CV F1 Macro: {final_cv_f1:.4f}")

print("\nFinal hyperparameters:")
for key, value in final_params.items():
    print("   ", key.replace("classifier__", ""), "=", value)

## 14. Final Test Evaluation

The held-out test set is used only now, after model selection and tuning.


In [ ]:
y_pred_B = final_model.predict(X_test_B)

track_B_results = {
    "Track": "Track B - Lifestyle Only",
    "Accuracy": accuracy_score(y_test, y_pred_B),
    "Precision Macro": precision_score(
        y_test, y_pred_B, average="macro", zero_division=0
    ),
    "Recall Macro": recall_score(
        y_test, y_pred_B, average="macro", zero_division=0
    ),
    "F1 Macro": f1_score(
        y_test, y_pred_B, average="macro", zero_division=0
    ),
}

track_B_results

### Track A Reference

Track A uses the same classifier type chosen for Track B — but with **default** hyperparameters,
while Track B was tuned. So the gap in the table understates the leak slightly. The like-for-like
number (both untuned, same folds) comes from the Section 11.3 table and is printed below.


In [ ]:
model_A = Pipeline([
    ("preprocess", clone(preprocess_A)),
    ("classifier", clone(models[best_model_name])),
])

model_A.fit(X_train_A, y_train)

y_pred_A = model_A.predict(X_test_A)

track_A_results = {
    "Track": "Track A - With Height/Weight/BMI",
    "Accuracy": accuracy_score(y_test, y_pred_A),
    "Precision Macro": precision_score(
        y_test, y_pred_A, average="macro", zero_division=0
    ),
    "Recall Macro": recall_score(
        y_test, y_pred_A, average="macro", zero_division=0
    ),
    "F1 Macro": f1_score(
        y_test, y_pred_A, average="macro", zero_division=0
    ),
}

final_results = pd.DataFrame([
    track_A_results,
    track_B_results,
])

display(final_results.round(4))

                                                                                             
majority_class = y_train.value_counts().idxmax()
majority_accuracy = (y_test == majority_class).mean()

like_for_like = (
    comparison.loc[comparison["Model"] == best_model_name, "Difference"].iloc[0]
)

print(f"Majority-class baseline accuracy : {majority_accuracy:.4f} "
      f"(always predicts '{majority_class}')")
print(f"Leakage gap, test (tuned B vs default A) : "
      f"{track_A_results['F1 Macro'] - track_B_results['F1 Macro']:.4f}")
print(f"Leakage gap, like-for-like CV (both default) : {like_for_like:.4f}")

### 14.1 Classification Report — Track B


In [ ]:
print("Final Track B model:", best_model_name)
print()

print(
    classification_report(
        y_test,
        y_pred_B,
        labels=CLASS_ORDER,
        zero_division=0,
    )
)

### 14.2 Confusion Matrices


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(19, 7))

cm_A = confusion_matrix(
    y_test,
    y_pred_A,
    labels=CLASS_ORDER
)

sns.heatmap(
    cm_A,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_ORDER,
    yticklabels=CLASS_ORDER,
    ax=axes[0],
    cbar=False,
)

axes[0].set_title("Track A")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].tick_params(axis="x", rotation=45)

cm_B = confusion_matrix(
    y_test,
    y_pred_B,
    labels=CLASS_ORDER
)

sns.heatmap(
    cm_B,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_ORDER,
    yticklabels=CLASS_ORDER,
    ax=axes[1],
    cbar=False,
)

axes[1].set_title("Track B")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

### 14.3 How far off are the mistakes?

The seven classes are ordered, so predicting `Overweight_Level_I` instead of
`Overweight_Level_II` is a much smaller error than predicting `Normal_Weight`. Plain accuracy
treats both the same; this does not.


In [ ]:
true_pos = y_test.map(class_position).to_numpy()
pred_pos = pd.Series(y_pred_B).map(class_position).to_numpy()
error_distance = np.abs(true_pos - pred_pos)

print("Track B error profile on the test set:")
print(f"   Exact match      : {(error_distance == 0).mean():.2%}")
print(f"   Within 1 class   : {(error_distance <= 1).mean():.2%}")
print(f"   Within 2 classes : {(error_distance <= 2).mean():.2%}")
print(f"   Mean error       : {error_distance.mean():.3f} classes")

## 15. Simple Overfitting Check

Compare training F1 with cross-validation F1 and test F1.


In [ ]:
train_pred = final_model.predict(X_train_B)

train_f1 = f1_score(
    y_train,
    train_pred,
    average="macro",
    zero_division=0,
)

cv_f1 = final_cv_f1
test_f1 = track_B_results["F1 Macro"]

overfitting_check = pd.DataFrame({
    "Score": [
        "Train F1 Macro",
        "CV F1 Macro",
        "Test F1 Macro",
    ],
    "Value": [
        train_f1,
        cv_f1,
        test_f1,
    ],
})

overfitting_check.round(4)

### Results

In [ ]:
train_cv_gap = train_f1 - cv_f1
cv_test_gap = cv_f1 - test_f1

print("Train - CV gap:", round(train_cv_gap, 4))
print("CV - Test gap :", round(cv_test_gap, 4))
print()

tree_like = best_model_name in ("Random Forest", "Decision Tree", "Gradient Boosting")

if train_f1 > 0.99 and tree_like:
    print("Train F1 is about 1.00. Tree models can fit the training data extremely closely.")
    print("The more important check is whether CV and test performance remain strong and similar.")
elif train_cv_gap > 0.10:
    print("Train performance is much higher than CV performance, which suggests overfitting.")
else:
    print("Train and CV performance are reasonably close.")

print()
if abs(cv_test_gap) > 0.05:
    print("CV and test differ by more than 5 F1 points, so interpret the estimate cautiously.")
else:
    print("CV and test agree within 5 F1 points.")

## 16. Save the Final Model

A `.pkl` on its own is not reusable — nothing records which columns the model expects. Saving a
small JSON next to it fixes that, and compression keeps the file to a sensible size.


In [ ]:
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

MODEL_PATH = ARTIFACT_DIR / "obesity_lifestyle_model.pkl"
INFO_PATH = ARTIFACT_DIR / "model_info.json"

                                                                                          
joblib.dump(final_model, MODEL_PATH, compress=3)

                                                                                           
model_info = {
    "model_name": best_model_name,
    "track": "B - lifestyle only (Height, Weight, BMI excluded on purpose)",
    "sklearn_version": sklearn.__version__,
    "feature_order": list(X_train_B.columns),
    "boolean_features": bool_B,
    "nominal_values": {c: sorted(df[c].astype(str).unique().tolist()) for c in nom_B},
    "ordinal_values": {"CAEC": CAEC_ORDER, "CALC": CALC_ORDER},
    "class_order": CLASS_ORDER,
    "hyperparameters": {k.replace("classifier__", ""): v for k, v in final_params.items()},
    "test_metrics": {k: float(v) for k, v in track_B_results.items() if k != "Track"},
}

INFO_PATH.write_text(json.dumps(model_info, indent=2))

print("Model saved to:", MODEL_PATH, f"({MODEL_PATH.stat().st_size / 1024:.1f} KB)")
print("Info saved to :", INFO_PATH)

                                                                   
reloaded = joblib.load(MODEL_PATH)
assert list(reloaded.predict(X_test_B.head(20))) == list(final_model.predict(X_test_B.head(20)))
print("Reload check passed: saved model gives identical predictions.")

## 17. Final Summary

The important numbers are printed together here after the notebook has been run.


In [ ]:
print("=" * 60)
print("OBESITY PREDICTION SUMMARY")
print("=" * 60)

print("Rows after cleaning:", len(df))
print("Train rows:", len(X_train_B))
print("Test rows:", len(X_test_B))

print()
print("BMI agreement with target:", f"{agreement:.2%}")
print("Share of misses one band away:", f"{(wrong == 1).mean():.1%}")
print("Likely synthetic rows:", f"{(~is_integral).mean():.1%}")

print()
print("Selected Track B model:", best_model_name, "(best tuned CV F1)")
print("Track B CV F1 Macro:", round(final_cv_f1, 4))
print("Track A F1 Macro:", round(track_A_results["F1 Macro"], 4))
print("Track B F1 Macro:", round(track_B_results["F1 Macro"], 4))
print("Track B Accuracy:", round(track_B_results["Accuracy"], 4))
print("Track B within 1 class:", f"{(error_distance <= 1).mean():.2%}")
print("Majority baseline accuracy:", round(majority_accuracy, 4))

print()
print("Model saved to:", MODEL_PATH)
print("=" * 60)

## Conclusion

This version keeps the machine-learning workflow clear and readable without custom helper
functions.

The key comparison is between:

- **Track A:** includes Height, Weight and BMI, so it can recover most of the obesity-label
  formula. Its high score is a sanity check, not a result.
- **Track B:** excludes those measurements and tests the real predictive value of the lifestyle
  variables.

Track B is the number to report. Two limitations belong next to it every time it is quoted:
most of the rows are synthetic, so a random split cannot fully separate a generated row from the
real rows it was interpolated from; and all lifestyle answers are self-reported.
